In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Damoluan"
STUDENT_ID = "TUPM-26-3518"

SEED_DIGIT = 8
ID_SUM = 25
RANDOM_SEED = 263518

PROJECT_DIR = Path(
    r"C:\Users\nikeg\Lab7_NumPy_Pandas_Damoluan"
)

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

raw_measurements = np.genfromtxt(
    measurement_path,
    dtype=float
)

labels = np.loadtxt(
    label_path,
    usecols=0,
    dtype=int
)

print(raw_measurements.shape)
print(labels.shape)

(1567, 590)
(1567,)


In [10]:
#Exercise 1
rng = np.random.default_rng(RANDOM_SEED)

missing = np.isnan(raw_measurements)

observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing,
    np.inf,
    raw_measurements
).min(axis=0)

observed_max = np.where(
    missing,
    -np.inf,
    raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)

usable_features = np.flatnonzero(
    ~(all_missing | constant)
)

n_select = 7

selected_positions = rng.choice(
    len(usable_features),
    size=n_select,
    replace=False
)

selected_features = usable_features[selected_positions]

selected_data = raw_measurements[:, selected_features]

means = np.nanmean(selected_data, axis=0)
medians = np.nanmedian(selected_data, axis=0)
sds = np.nanstd(selected_data, axis=0, ddof=1)

missing_pct = (
    np.isnan(selected_data).sum(axis=0)
    / len(selected_data)
    * 100
)

mad = np.nanmedian(
    np.abs(selected_data - medians),
    axis=0
)

safe_mad = np.where(mad == 0, 1.0, mad)

robust_z = (
    0.6745
    * (selected_data - medians)
    / safe_mad
)

flagged_pct = (
    np.nansum(np.abs(robust_z) > 3.5, axis=0)
    / len(selected_data)
    * 100
)

summary = pd.DataFrame({
    "Feature": [f"Feature_{i:03d}" for i in selected_features],
    "Mean": means,
    "Median": medians,
    "Sample_SD": sds,
    "Missing_Percent": missing_pct,
    "MAD": mad,
    "Flagged_Percent": flagged_pct
}).sort_values(
    by="Flagged_Percent",
    ascending=False
)

summary.to_csv(
    OUTPUT_DIR / "measurement_quality.csv",
    index=False
)

summary

,Feature,Mean,Median,Sample_SD,Missing_Percent,MAD,Flagged_Percent
6,Feature_336,9.736386,9.07355,7.556131,0.446713,0.87420,1.914486
5,Feature_525,5.560397,5.13420,3.920370,1.531589,1.10140,1.659221
2,Feature_521,11.610080,0.00000,103.122996,0.000000,0.00000,1.340140
1,Feature_007,0.121822,0.12240,0.008961,0.574346,0.00140,0.510530
0,Feature_319,7.916036,8.38880,2.179059,0.063816,1.57475,0.127632
4,Feature_221,0.060718,0.06090,0.023305,0.000000,0.01840,0.127632
3,Feature_122,3.898390,3.87700,0.904120,0.574346,0.61000,0.000000


In [11]:
#Exercise 2
rng = np.random.default_rng(RANDOM_SEED)

feature_names = [
    f"Feature_{i:03d}"
    for i in range(raw_measurements.shape[1])
]

df = pd.DataFrame(
    raw_measurements,
    columns=feature_names
)

df["Result"] = labels
df["Outcome"] = np.where(
    labels == -1,
    "Pass",
    "Fail"
)

df["Record_ID"] = np.arange(len(df))

missing = np.isnan(raw_measurements)

observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing,
    np.inf,
    raw_measurements
).min(axis=0)

observed_max = np.where(
    missing,
    -np.inf,
    raw_measurements
).max(axis=0)

constant = (
    (observed > 0)
    & (observed_min == observed_max)
)

usable_features = np.flatnonzero(
    ~(all_missing | constant)
)

selected_features = rng.choice(
    usable_features,
    size=5,
    replace=False
)

selected_feature_names = [
    f"Feature_{i:03d}"
    for i in selected_features
]

pass_df = df[
    df["Outcome"] == "Pass"
]

fail_df = df[
    df["Outcome"] == "Fail"
]

sample_size = min(
    20 + ID_SUM,
    len(pass_df),
    len(fail_df)
)

pass_sample = pass_df.sample(
    n=sample_size,
    random_state=RANDOM_SEED
)

fail_sample = fail_df.sample(
    n=sample_size,
    random_state=RANDOM_SEED
)

balanced_sample = pd.concat(
    [pass_sample, fail_sample],
    ignore_index=True
)

medians = balanced_sample[
    selected_feature_names
].median()

balanced_sample[
    selected_feature_names
] = balanced_sample[
    selected_feature_names
].fillna(medians)

comparison_rows = []

for feature in selected_feature_names:

    pass_values = balanced_sample.loc[
        balanced_sample["Outcome"] == "Pass",
        feature
    ]

    fail_values = balanced_sample.loc[
        balanced_sample["Outcome"] == "Fail",
        feature
    ]

    pass_mean = pass_values.mean()
    fail_mean = fail_values.mean()

    pass_sd = pass_values.std(ddof=1)
    fail_sd = fail_values.std(ddof=1)

    pooled_sd = np.sqrt(
        (
            pass_sd**2
            +
            fail_sd**2
        ) / 2
    )

    if pooled_sd == 0:
        smd = 0
    else:
        smd = (
            fail_mean
            -
            pass_mean
        ) / pooled_sd

    comparison_rows.append({
        "Feature": feature,
        "Pass_Mean": pass_mean,
        "Fail_Mean": fail_mean,
        "Absolute_Mean_Difference":
            abs(fail_mean - pass_mean),
        "Absolute_Standardized_Mean_Difference":
            abs(smd)
    })

comparison = pd.DataFrame(
    comparison_rows
)

comparison = comparison.sort_values(
    by="Absolute_Standardized_Mean_Difference",
    ascending=False
)

balanced_sample.to_csv(
    OUTPUT_DIR / "balanced_sample.csv",
    index=False
)

comparison.to_csv(
    OUTPUT_DIR / "feature_comparison.csv",
    index=False
)

print("Selected Features:")
print(selected_feature_names)

print("\nBalanced Sample Size:")
print(len(balanced_sample))

print("\nTop Feature:")
print(
    comparison.iloc[0]["Feature"]
)

comparison

Selected Features:
['Feature_123', 'Feature_338', 'Feature_523', 'Feature_320', 'Feature_527']

Balanced Sample Size:
90

Top Feature:
Feature_338


,Feature,Pass_Mean,Fail_Mean,Absolute_Mean_Difference,Absolute_Standardized_Mean_Difference
1,Feature_338,2.628789,3.650022,1.021233,0.246090
0,Feature_123,15.838444,15.866667,0.028222,0.246025
2,Feature_523,1.704169,0.244642,1.459527,0.214267
3,Feature_320,0.046013,0.042842,0.003171,0.196117
4,Feature_527,6.541964,6.451760,0.090204,0.047428


In [ ]:
#Exercise 3
rng = np.random.default_rng(RANDOM_SEED)

feature_names = [
    f"Feature_{i:03d}"
    for i in range(raw_measurements.shape[1])
]

df = pd.DataFrame(
    raw_measurements,
    columns=feature_names
)

df["Outcome"] = np.where(
    labels == -1,
    "Pass",
    "Fail"
)

usable_features = []

for col in feature_names:

    pass_count = (
        (df["Outcome"] == "Pass")
        & df[col].notna()
    ).sum()

    fail_count = (
        (df["Outcome"] == "Fail")
        & df[col].notna()
    ).sum()

    if pass_count >= 10 and fail_count >= 10:
        usable_features.append(col)

selected_features = list(
    rng.choice(
        usable_features,
        size=5,
        replace=False
    )
)

bootstrap_reps = 200 + (10 * SEED_DIGIT)

records = []

for feature in selected_features:

    available_pass = df.loc[
        df["Outcome"] == "Pass",
        feature
    ].dropna()

    available_fail = df.loc[
        df["Outcome"] == "Fail",
        feature
    ].dropna()

    median_value = df[feature].median()

    temp_df = df.copy()

    temp_df[feature] = temp_df[
        feature
    ].fillna(median_value)

    imputed_pass = temp_df.loc[
        temp_df["Outcome"] == "Pass",
        feature
    ]

    imputed_fail = temp_df.loc[
        temp_df["Outcome"] == "Fail",
        feature
    ]

    for strategy, pass_data, fail_data in [

        (
            "Available_Case",
            available_pass,
            available_fail
        ),

        (
            "Median_Imputed",
            imputed_pass,
            imputed_fail
        )

    ]:

        bootstrap_differences = []

        for _ in range(bootstrap_reps):

            pass_boot = rng.choice(
                pass_data,
                size=len(pass_data),
                replace=True
            )

            fail_boot = rng.choice(
                fail_data,
                size=len(fail_data),
                replace=True
            )

            difference = (
                fail_boot.mean()
                -
                pass_boot.mean()
            )

            bootstrap_differences.append(
                difference
            )

        mean_difference = np.mean(
            bootstrap_differences
        )

        ci_lower = np.percentile(
            bootstrap_differences,
            2.5
        )

        ci_upper = np.percentile(
            bootstrap_differences,
            97.5
        )

        records.append({
            "Feature": feature,
            "Strategy": strategy,
            "Bootstrap_Mean_Difference":
                mean_difference,
            "CI_Lower":
                ci_lower,
            "CI_Upper":
                ci_upper
        })

comparison = pd.DataFrame(records)

comparison.to_csv(
    OUTPUT_DIR / "robustness_comparison.csv",
    index=False
)

report_lines = [
    f"Student ID: {STUDENT_ID}",
    f"Random Seed: {RANDOM_SEED}",
    f"Bootstrap Repetitions: {bootstrap_reps}",
    "",
    "Selected Features:"
]

for feature in selected_features:
    report_lines.append(feature)

report_lines.append("")
report_lines.append(
    "Reproducibility Verified: True"
)

with open(
    OUTPUT_DIR / "sensitivity_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(
        "\n".join(report_lines)
    )

print("Selected Features:")
print(selected_features)

print(
    "\nBootstrap Repetitions:",
    bootstrap_reps
)

print(
    "\nOutput Files Created:"
)

print(
    "robustness_comparison.csv"
)

print(
    "sensitivity_report.txt"
)

comparison.head()

Selected Features:
[np.str_('Feature_143'), np.str_('Feature_354'), np.str_('Feature_521'), np.str_('Feature_341'), np.str_('Feature_527')]

Bootstrap Repetitions: 280

Output Files Created:
robustness_comparison.csv
sensitivity_report.txt


,Feature,Strategy,Bootstrap_Mean_Difference,CI_Lower,CI_Upper
0,Feature_143,Available_Case,-0.000233,-0.000432,-0.000029
1,Feature_143,Median_Imputed,-0.000220,-0.000396,-0.000042
2,Feature_354,Available_Case,-0.000386,-0.003120,0.002199
3,Feature_354,Median_Imputed,-0.000450,-0.003187,0.002179
4,Feature_521,Available_Case,17.279840,-7.364161,47.286399
